# Pemetaan Kualitas Air Pesisir Provinsi Banten Per Kecamatan
### Analisis Spasial Komparatif Berbasis openEO & Sentinel-2

Notebook ini digunakan untuk menganalisis kualitas air pesisir per kecamatan di Banten (2017 vs 2026) menggunakan citra Sentinel-2. Parameter kualitas air yang dideteksi:
1. **Kekeruhan (NDTI - Turbidity)**: Kriteria sedimentasi pantai.
2. **Klorofil-a (NDCI - Chlorophyll-a)**: Kriteria nutrien/blooming alga.
3. **TSS (Total Suspended Solids) & CDOM**: Indikator material tersuspensi dan materi organik terlarut.
4. **Model Klasifikasi Supervised**: Random Forest Classifier dilatih secara global untuk Banten untuk mengklasifikasi status kualitas air (**SEHAT**, **SEDANG**, **TIDAK SEHAT**).

---
## 1. Setup Lingkungan & Import Library

In [ ]:
# Install dependensi eksternal (jalankan jika belum terinstal pada environment saat ini)
# !pip install openeo rasterio rioxarray xarray geopandas shapely fiona folium scikit-learn matplotlib pandas numpy joblib

import os
import sys
import glob
import shutil
import tempfile
import time
import json
import math
import joblib
from datetime import datetime

import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import rioxarray
import folium
from folium import plugins
from shapely.geometry import mapping
from sklearn.ensemble import RandomForestClassifier

print("Libraries successfully imported.")
print(f"Execution timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

---
## 2. Koneksi dan Autentikasi openEO

In [ ]:
print("Connecting to Copernicus Data Space Ecosystem (openEO)...")
try:
    import openeo
    connection = openeo.connect('https://openeo.dataspace.copernicus.eu')
    connection.authenticate_oidc()
    print("Authentication successful. openEO session initialized.")
except Exception as e:
    print(f"Warning: openEO connection failed ({e}). Kelanjutan unduhan openEO mungkin terganggu.")

---
## 3. Konfigurasi Parameter Kualitas Air

In [ ]:
# Parameter Temporal
YEAR_T1 = 2017          # Tahun dasar (baseline)
YEAR_T2 = 2026          # Tahun pembanding (terkini)

# Parameter Musiman (target musim kemarau untuk reduksi tutupan awan)
DRY_SEASON_START = 5    # Mei
DRY_SEASON_END = 10     # Oktober

SCALE_EXPORT = 30       # Resolusi ekspor composite (30 meter)
GADM_PATH = "data/gadm41_IDN.gpkg"
COMPOSITE_DIR = "data/composites"
OUTPUT_DIR = "output"

# Batas Klasifikasi Kualitas Air (Threshold Fisik-Kimiawi)
NDTI_TURBID_THRESHOLD = 0.05       # NDTI > 0.05 -> Sangat Keruh (Tidak Sehat)
NDCI_BLOOM_THRESHOLD = 0.08        # NDCI > 0.08 -> Blooming Alga (Tidak Sehat)
NDCI_LOW_THRESHOLD = -0.02         # NDCI <= -0.02 -> Sangat Jernih/Rendah Nutrisi
NDTI_CLEAR_THRESHOLD = -0.05       # NDTI <= -0.05 -> Sangat Jernih

print("Configuration parameters successfully loaded.")

---
## 4. Ekstraksi Wilayah Kecamatan Pesisir Banten (Batas Laut 3 km)

In [ ]:
def load_banten_coastal_kecamatans(gadm_path=GADM_PATH, buffer_meters=3000, min_water_area_ha=10.0, return_land_geom=False):
    print("Mengekstrak data batas wilayah kecamatan pesisir Banten...")
    if not os.path.exists(gadm_path):
        raise FileNotFoundError(f"File GADM tidak ditemukan di {gadm_path}. Silakan jalankan data_downloader.py terlebih dahulu.")
        
    gdf = gpd.read_file(gadm_path, layer="ADM_ADM_3")
    
    # Saring Banten dan tetangganya
    banten = gdf[gdf["NAME_1"] == "Banten"].copy()
    neighbors = gdf[gdf["NAME_1"].isin(["Banten", "Jawa Barat", "Jakarta Raya", "Dki Jakarta"])].copy()
    
    # Proyeksikan ke UTM 48S (EPSG:32748) untuk operasi buffer dalam meter
    banten_utm = banten.to_crs(epsg=32748)
    neighbors_utm = neighbors.to_crs(epsg=32748)
    land_geom = neighbors_utm.union_all()
    
    coastal_rows = []
    for idx, row in banten_utm.iterrows():
        geom = row.geometry
        name_3 = row["NAME_3"]
        name_2 = row["NAME_2"]
        gid_3 = row["GID_3"]
        
        # Buat buffer 3 km
        buffered = geom.buffer(buffer_meters)
        
        # Kurangkan daratan untuk mendapatkan zona perairan pesisir
        water_zone = buffered.difference(land_geom)
        water_area_ha = water_zone.area / 10000.0
        
        if water_area_ha >= min_water_area_ha:
            coastal_rows.append({
                "GID_3": gid_3,
                "Kabupaten_Kota": name_2,
                "Kecamatan": name_3,
                "water_area_ha": water_area_ha,
                "land_geom": geom,
                "geometry": water_zone
            })
            
    coastal_gdf = gpd.GeoDataFrame(coastal_rows, crs="EPSG:32748")
    coastal_gdf = coastal_gdf.to_crs(epsg=4326)
    coastal_gdf["land_geom"] = coastal_gdf["land_geom"].apply(lambda g: gpd.GeoSeries([g], crs="EPSG:32748").to_crs(epsg=4326).iloc[0])
    
    print(f"Ditemukan {len(coastal_gdf)} kecamatan pesisir di Banten.")
    if return_land_geom:
        return coastal_gdf, land_geom
    return coastal_gdf

def load_banten_coastal_beaches(land_geom_utm, buffer_meters=1000):
    print("\nMengekstrak data batas wilayah pantai pesisir Banten...")
    beaches = [
        {"name": "Pantai Anyer", "Kecamatan": "Anyar", "Kabupaten_Kota": "Serang", "lat": -6.0465, "lon": 105.8850},
        {"name": "Pantai Carita", "Kecamatan": "Carita", "Kabupaten_Kota": "Pandeglang", "lat": -6.1305, "lon": 105.8427},
        {"name": "Pantai Tanjung Lesung", "Kecamatan": "Panimbang", "Kabupaten_Kota": "Pandeglang", "lat": -6.4785, "lon": 105.6565},
        {"name": "Pantai Sawarna", "Kecamatan": "Bayah", "Kabupaten_Kota": "Lebak", "lat": -6.9930, "lon": 106.3180},
        {"name": "Pantai Bagedur", "Kecamatan": "Malingping", "Kabupaten_Kota": "Lebak", "lat": -6.9038, "lon": 106.0125},
        {"name": "Pantai Karang Bolong", "Kecamatan": "Cinangka", "Kabupaten_Kota": "Serang", "lat": -6.1082, "lon": 105.8569},
        {"name": "Pantai Ciputih", "Kecamatan": "Sumur", "Kabupaten_Kota": "Pandeglang", "lat": -6.6575, "lon": 105.5180},
        {"name": "Pantai Pulau Umang", "Kecamatan": "Sumur", "Kabupaten_Kota": "Pandeglang", "lat": -6.64065, "lon": 105.58436},
        {"name": "Pantai Sambolo", "Kecamatan": "Anyar", "Kabupaten_Kota": "Serang", "lat": -6.0712, "lon": 105.8812},
        {"name": "Pantai Pasir Putih Sirih", "Kecamatan": "Anyar", "Kabupaten_Kota": "Serang", "lat": -6.0825, "lon": 105.8805},
        {"name": "Pantai Marbella", "Kecamatan": "Anyar", "Kabupaten_Kota": "Serang", "lat": -6.0620, "lon": 105.8825},
        {"name": "Pantai Florida Indah", "Kecamatan": "Cinangka", "Kabupaten_Kota": "Serang", "lat": -6.1345, "lon": 105.8670},
        {"name": "Pantai Jambu", "Kecamatan": "Cinangka", "Kabupaten_Kota": "Serang", "lat": -6.1158, "lon": 105.8640},
        {"name": "Pantai Lontar", "Kecamatan": "Tirtayasa", "Kabupaten_Kota": "Serang", "lat": -5.96884, "lon": 106.29646},
        {"name": "Pantai Tanjung Pasir", "Kecamatan": "Teluknaga", "Kabupaten_Kota": "Tangerang", "lat": -6.0150, "lon": 106.6850},
        {"name": "Pantai Tanjung Kait", "Kecamatan": "Mauk", "Kabupaten_Kota": "Tangerang", "lat": -6.0195, "lon": 106.4520},
        {"name": "Pantai Binuangeun", "Kecamatan": "Wanasalam", "Kabupaten_Kota": "Lebak", "lat": -6.8290, "lon": 105.9030},
        {"name": "Pantai Karang Taraje", "Kecamatan": "Bayah", "Kabupaten_Kota": "Lebak", "lat": -6.9912, "lon": 106.3312},
        {"name": "Pantai Sangiang", "Kecamatan": "Anyar", "Kabupaten_Kota": "Serang", "lat": -5.9535, "lon": 105.8565},
        {"name": "Pantai Pasir Putih Florida", "Kecamatan": "Cinangka", "Kabupaten_Kota": "Serang", "lat": -6.1265, "lon": 105.8645},
        {"name": "Pantai Karang Songsong", "Kecamatan": "Cihara", "Kabupaten_Kota": "Lebak", "lat": -6.88395, "lon": 106.11112},
        {"name": "Pantai Pulau Cangkir", "Kecamatan": "Kronjo", "Kabupaten_Kota": "Tangerang", "lat": -6.00889, "lon": 106.42000},
        {"name": "Pantai Pasir Putih Cihara", "Kecamatan": "Cihara", "Kabupaten_Kota": "Lebak", "lat": -6.84643, "lon": 106.06923},
        {"name": "Pantai Tanjung Layar", "Kecamatan": "Bayah", "Kabupaten_Kota": "Lebak", "lat": -6.99431, "lon": 106.30716},
        {"name": "Pantai Caringin", "Kecamatan": "Labuan", "Kabupaten_Kota": "Pandeglang", "lat": -6.35329, "lon": 105.82301}
    ]
    from shapely.geometry import Point
    df = pd.DataFrame(beaches)
    geometry = [Point(xy) for xy in zip(df['lon'], df['lat'])]
    beaches_gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")
    beaches_utm = beaches_gdf.to_crs(epsg=32748)
    
    coastal_beaches = []
    for idx, row in beaches_utm.iterrows():
        buffered = row.geometry.buffer(buffer_meters)
        water_zone = buffered.difference(land_geom_utm)
        if not water_zone.is_empty:
            coastal_beaches.append({
                "Pantai": row["name"],
                "Kecamatan": row["Kecamatan"],
                "Kabupaten_Kota": row["Kabupaten_Kota"],
                "latitude": row["lat"],
                "longitude": row["lon"],
                "geometry": water_zone
            })
    coastal_beaches_gdf = gpd.GeoDataFrame(coastal_beaches, crs="EPSG:32748")
    coastal_beaches_gdf = coastal_beaches_gdf.to_crs(epsg=4326)
    print(f"Ditemukan {len(coastal_beaches_gdf)} lokasi pantai pesisir Banten yang siap dianalisis.")
    return coastal_beaches_gdf

INDUSTRIES = [
    {"nama": "PT Krakatau Steel", "tipe": "Baja/Logam", "latitude": -6.0048, "longitude": 106.0148},
    {"nama": "PT Chandra Asri Petrochemical", "tipe": "Petrokimia", "latitude": -6.0200, "longitude": 106.0050},
    {"nama": "PT Asahimas Chemical", "tipe": "Kimia", "latitude": -6.0120, "longitude": 106.0200},
    {"nama": "PLTU Suralaya", "tipe": "Pembangkit Listrik", "latitude": -5.9320, "longitude": 105.9440},
    {"nama": "PT Indah Kiat Pulp & Paper (Merak)", "tipe": "Pulp & Paper", "latitude": -5.9550, "longitude": 106.0000},
    {"nama": "PT Indonesia Power Suralaya", "tipe": "Energi", "latitude": -5.9350, "longitude": 105.9460},
    {"nama": "PT Banten Energy", "tipe": "Energi", "latitude": -6.0450, "longitude": 105.9750},
    {"nama": "Pelabuhan Merak", "tipe": "Pelabuhan", "latitude": -5.9350, "longitude": 106.0000},
    {"nama": "PT Lotte Chemical Indonesia", "tipe": "Kimia", "latitude": -6.0380, "longitude": 106.0100},
    {"nama": "PLTU Labuan (Banten 2)", "tipe": "Pembangkit Listrik", "latitude": -6.3660, "longitude": 105.8180},
    {"nama": "PT Indocement Tunggal Prakarsa (Bayah)", "tipe": "Semen", "latitude": -6.9500, "longitude": 106.2600},
    {"nama": "Pelabuhan Ciwandan", "tipe": "Pelabuhan/Logistik", "latitude": -6.0350, "longitude": 105.9600},
    {"nama": "PT Sulfindo Adiusaha", "tipe": "Kimia", "latitude": -6.0150, "longitude": 106.0080},
    {"nama": "Pelabuhan Perikanan Karangantu", "tipe": "Perikanan", "latitude": -6.0310, "longitude": 106.1700}
]

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1_rad = math.radians(lat1)
    lat2_rad = math.radians(lat2)
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat / 2) ** 2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return round(R * c, 2)

def get_dampak_category(distance_km):
    if distance_km < 5:
        return "TINGGI"
    elif distance_km <= 15:
        return "SEDANG"
    else:
        return "RENDAH"

def find_nearest_industries(lat, lon, top_n=3):
    distances = []
    for industry in INDUSTRIES:
        dist = haversine(lat, lon, industry["latitude"], industry["longitude"])
        distances.append({
            "nama": industry["nama"],
            "tipe": industry["tipe"],
            "jarak_km": dist,
            "lat": industry["latitude"],
            "lon": industry["longitude"]
        })
    distances.sort(key=lambda x: x["jarak_km"])
    return distances[:top_n]

def _build_industry_text(stats):
    industri = stats.get("industri_terdekat")
    jarak = stats.get("jarak_industri_km")
    kategori = stats.get("kategori_dampak_industri")
    tipe = stats.get("tipe_industri", "")
    if not industri or jarak is None:
        return ""
    if kategori == "TINGGI":
        return (
            f" Lokasi ini berada dalam radius dampak **TINGGI** dari fasilitas industri {industri} ({tipe}) "
            f"yang berjarak hanya **{jarak} km**, sehingga potensi kontribusi polutan industri terhadap penurunan kualitas air sangat signifikan."
        )
    elif kategori == "SEDANG":
        return (
            f" Terdapat fasilitas industri {industri} ({tipe}) dalam radius **{jarak} km** "
            f"dengan kategori dampak **SEDANG**, yang berpotensi turut memengaruhi kondisi kualitas perairan."
        )
    else:
        return (
            f" Industri terdekat adalah {industri} ({tipe}) berjarak **{jarak} km** "
            f"dengan kategori dampak **RENDAH**."
        )

def generate_beach_explanation(beach_name, kec_name, status, ndti, ndci, stats):
    industry_text = _build_industry_text(stats)
    if status == "SEHAT":
        return f"Kualitas air di {beach_name} ({kec_name}) tergolong **SEHAT** (Bersih). Kondisi perairan pantai sangat bersih dengan kekeruhan rendah (NDTI: {ndti:.4f}) and klorofil-a (NDCI: {ndci:.4f}) yang normal, menjadikannya sangat aman dan nyaman untuk kegiatan pariwisata atau berenang.{industry_text}"
    elif status == "SEDANG":
        return f"Kualitas air di {beach_name} ({kec_name}) berada dalam kondisi **SEDANG**. Perairan pantai cukup bersih namun tingkat kekeruhan (NDTI: {ndti:.4f}) atau klorofil-a (NDCI: {ndci:.4f}) menunjukkan nilai ambang batas wajar. Pengunjung dihimbau tetap menjaga kebersihan pantai sekitar.{industry_text}"
    else:
        reasons = []
        if ndti > 0.05: reasons.append(f"tingginya kekeruhan air (NDTI: {ndti:.4f}) akibat limpasan sedimen darat")
        if ndci > 0.08: reasons.append(f"kadar klorofil-a yang tinggi (NDCI: {ndci:.4f}) yang menandakan penumpukan nutrien/blooming alga")
        reason_str = " dan ".join(reasons) if reasons else "penurunan baku mutu air laut pesisir"
        return f"Kualitas air di {beach_name} ({kec_name}) tergolong **TIDAK SEHAT** (Tercemar). Analisis menunjukkan {reason_str}. Disarankan untuk membatasi aktivitas kontak langsung seperti berenang di sekitar perairan pantai ini.{industry_text}"

coastal_gdf, land_geom_utm = load_banten_coastal_kecamatans(return_land_geom=True)
coastal_gdf[["Kabupaten_Kota", "Kecamatan", "water_area_ha"]].head(10)

---
## 5. Fungsi openEO Composite Downloader & Indeks Spektral Kualitas Air

In [ ]:
def get_sentinel2_water_composite_job(connection, bbox, year, scale_export=SCALE_EXPORT):
    start_date = f"{year}-{DRY_SEASON_START:02d}-01"
    end_date   = f"{year}-{DRY_SEASON_END:02d}-31"

    extent = {
        "west": bbox[0], "south": bbox[1],
        "east": bbox[2], "north": bbox[3],
        "crs": "EPSG:4326",
    }

    cube = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=extent,
        temporal_extent=[start_date, end_date],
        bands=["B02", "B03", "B04", "B05", "B08", "B11", "SCL"],
    )

    scl = cube.band("SCL")
    cloud_mask = ~(
        (scl == 1) | (scl == 3) | (scl == 8) |
        (scl == 9) | (scl == 10) | (scl == 11)
    )
    masked_cube = cube.mask(cloud_mask)

    res_deg = scale_export / 111320.0
    resampled = masked_cube.resample_spatial(
        resolution=res_deg, projection=4326, method="bilinear",
    )
    composite = resampled.reduce_dimension(reducer="median", dimension="t")
    return composite.filter_bands(["B02", "B03", "B04", "B05", "B08", "B11"])

def download_banten_composite(connection, bbox, year, output_path, scale_export=SCALE_EXPORT):
    if os.path.exists(output_path):
        print(f"Composite untuk tahun {year} sudah tersedia di: {output_path}")
        return
    print(f"Mengajukan batch job openEO untuk composite Banten tahun {year}...")
    cube = get_sentinel2_water_composite_job(connection, bbox, year, scale_export)
    cube = cube.save_result("GTiff")

    job = cube.create_job(title=f"banten_water_quality_{year}")
    job.start_and_wait()

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with tempfile.TemporaryDirectory() as tmpdir:
        job.download_results(tmpdir)
        downloaded = glob.glob(os.path.join(tmpdir, "*.tif"))
        if downloaded:
            shutil.move(downloaded[0], output_path)
            print(f"Composite berhasil disimpan ke: {output_path}")
        else:
            raise RuntimeError("Gagal mengunduh file dari openEO.")

def load_composite_as_dataset(tif_path):
    raw = rioxarray.open_rasterio(tif_path)
    filename = os.path.basename(tif_path)
    if "Water" in filename:
        band_names = ["B02", "B03", "B04", "B05", "B08", "B11"]
        print(f"Memuat composite dengan B05 (Red Edge): {filename}")
    else:
        band_names = ["B02", "B03", "B04", "B08", "B11", "B12"]
        print(f"Memuat composite standar (menggunakan fallback): {filename}")
        
    ds = xr.Dataset({
        name: raw.sel(band=i + 1).drop_vars("band")
        for i, name in enumerate(band_names)
    })
    ds = ds.rio.write_crs("EPSG:4326")
    return ds

def calculate_water_quality_indices(ds):
    blue  = ds["B02"]
    green = ds["B03"]
    red   = ds["B04"]
    eps   = 1e-10

    if "B11" in ds:
        swir1 = ds["B11"]
        ds["MNDWI"] = (green - swir1) / (green + swir1 + eps)
    else:
        nir = ds["B08"]
        ds["MNDWI"] = (green - nir) / (green + nir + eps)

    ds["NDTI"] = (green - red) / (green + red + eps)

    if "B05" in ds:
        red_edge = ds["B05"]
        ds["NDCI"] = (red_edge - red) / (red_edge + red + eps)
    else:
        ds["NDCI"] = (green - blue) / (green + blue + eps)

    ds["TSS"] = red / (green + eps)
    ds["CDOM"] = green / (blue + eps)
    return ds

---
## 6. Algoritma Klasifikasi Kualitas Air & Analisis Lokal

In [ ]:
def classify_water_pixels_threshold(ds):
    mndwi = ds["MNDWI"].values
    ndti  = ds["NDTI"].values
    ndci  = ds["NDCI"].values
    
    water_mask = (mndwi > 0.0) & (~np.isnan(mndwi))
    classes = np.zeros_like(mndwi, dtype=np.uint8)
    
    # 1. Kekeruhan Tinggi
    turbid_mask = water_mask & (ndti > NDTI_TURBID_THRESHOLD)
    classes[turbid_mask] = 1
    
    # 2. Blooming Alga
    bloom_mask = water_mask & (ndci > NDCI_BLOOM_THRESHOLD) & (classes == 0)
    classes[bloom_mask] = 2
    
    # 3. Rendah Nutrisi
    clear_low_nut_mask = water_mask & (ndci <= NDCI_LOW_THRESHOLD) & (ndti <= NDTI_CLEAR_THRESHOLD) & (classes == 0)
    classes[clear_low_nut_mask] = 3
    
    # 4. Kondisi Optimum (Sehat)
    optimum_mask = (
        water_mask & 
        (ndti > NDTI_CLEAR_THRESHOLD) & (ndti <= NDTI_TURBID_THRESHOLD) &
        (ndci > NDCI_LOW_THRESHOLD) & (ndci <= NDCI_BLOOM_THRESHOLD) &
        (classes == 0)
    )
    classes[optimum_mask] = 4
    
    # 5. Kondisi Sedang / Lainnya
    sedang_mask = water_mask & (classes == 0)
    classes[sedang_mask] = 5
    return classes

def train_water_quality_rf(ds, threshold_classes):
    water_mask = threshold_classes > 0
    features_bands = ["NDVI" if "NDVI" in ds else "B08", "B02", "B03", "B04", "NDTI", "NDCI", "TSS", "CDOM"]
    features_bands = [f for f in features_bands if f in ds]
    
    features_list = [ds[band].values[water_mask] for band in features_bands]
    X = np.stack(features_list, axis=1)
    y = threshold_classes[water_mask]
    
    valid = ~np.isnan(X).any(axis=1)
    X = X[valid]
    y = y[valid]
    
    if len(X) < 100:
        return None
        
    rf = RandomForestClassifier(n_estimators=100, min_samples_leaf=5, random_state=42)
    rf.fit(X, y)
    
    all_features = [ds[band].values for band in features_bands]
    stacked = np.stack(all_features, axis=-1)
    y_dim, x_dim, n_feats = stacked.shape
    flat_all = stacked.reshape(-1, n_feats)
    flat_water_mask = water_mask.reshape(-1)
    valid_flat = flat_water_mask & (~np.isnan(flat_all).any(axis=1))
    
    predicted_flat = np.zeros(y_dim * x_dim, dtype=np.uint8)
    if np.sum(valid_flat) > 0:
        predicted_flat[valid_flat] = rf.predict(flat_all[valid_flat])
        
    return predicted_flat.reshape(y_dim, x_dim)

def compute_pixel_area_ha(ds):
    res_x = abs(ds.rio.transform()[0])
    res_y = abs(ds.rio.transform()[4])
    return (res_x * 111320.0) * (res_y * 111320.0) / 10000.0

def analyze_kecamatan_water_quality(ds, geom, pixel_area_ha):
    try:
        clipped_ds = ds.rio.clip([geom], crs="EPSG:4326", all_touched=True)
    except Exception as e:
        return None
        
    clipped_ds = calculate_water_quality_indices(clipped_ds)
    threshold_classes = classify_water_pixels_threshold(clipped_ds)
    final_classes = train_water_quality_rf(clipped_ds, threshold_classes)
    if final_classes is None:
        final_classes = threshold_classes
        
    total_water_pixels = np.sum(final_classes > 0)
    if total_water_pixels == 0:
        return None
        
    unhealthy_pixels = np.sum((final_classes == 1) | (final_classes == 2))
    healthy_pixels = np.sum(final_classes == 4)
    moderate_pixels = np.sum((final_classes == 3) | (final_classes == 5))
    
    water_area_ha = float(total_water_pixels * pixel_area_ha)
    healthy_ha = float(healthy_pixels * pixel_area_ha)
    moderate_ha = float(moderate_pixels * pixel_area_ha)
    unhealthy_ha = float(unhealthy_pixels * pixel_area_ha)
    
    pct_healthy = (healthy_ha / water_area_ha * 100) if water_area_ha > 0 else 0
    pct_moderate = (moderate_ha / water_area_ha * 100) if water_area_ha > 0 else 0
    pct_unhealthy = (unhealthy_ha / water_area_ha * 100) if water_area_ha > 0 else 0
    
    water_mask = final_classes > 0
    mean_ndti = float(np.nanmean(clipped_ds["NDTI"].values[water_mask]))
    mean_ndci = float(np.nanmean(clipped_ds["NDCI"].values[water_mask]))
    mean_tss = float(np.nanmean(clipped_ds["TSS"].values[water_mask]))
    mean_cdom = float(np.nanmean(clipped_ds["CDOM"].values[water_mask]))
    
    if pct_healthy > 50.0:
        status = "SEHAT"
    elif pct_unhealthy > 30.0:
        status = "TIDAK SEHAT"
    else:
        status = "SEDANG"
        
    return {
        "water_area_ha": round(water_area_ha, 2),
        "healthy_ha": round(healthy_ha, 2),
        "moderate_ha": round(moderate_ha, 2),
        "unhealthy_ha": round(unhealthy_ha, 2),
        "pct_healthy": round(pct_healthy, 1),
        "pct_moderate": round(pct_moderate, 1),
        "pct_unhealthy": round(pct_unhealthy, 1),
        "mean_ndti": round(mean_ndti, 4),
        "mean_ndci": round(mean_ndci, 4),
        "mean_tss": round(mean_tss, 4),
        "mean_cdom": round(mean_cdom, 4),
        "status": status
    }

---
## 7. Eksekusi Pencarian / Pengunduhan Composite Sentinel-2

In [ ]:
combined_geom = coastal_gdf.union_all()
bbox = combined_geom.bounds
print(f"Bbox perairan pesisir Banten: {bbox}")

path_t1 = os.path.join(COMPOSITE_DIR, f"Banten_Water_{YEAR_T1}.tif")
path_t2 = os.path.join(COMPOSITE_DIR, f"Banten_Water_{YEAR_T2}.tif")

# Mode download (True = Lewati download, gunakan fallback data lokal yang sudah ada)
SKIP_OPENEO = False 

if not SKIP_OPENEO:
    try:
        download_banten_composite(connection, bbox, YEAR_T1, path_t1, SCALE_EXPORT)
        download_banten_composite(connection, bbox, YEAR_T2, path_t2, SCALE_EXPORT)
    except Exception as exc:
        print(f"Error openEO: {exc}. Menggunakan local fallback.")

---
## 8. Eksekusi Loop Analisis Kualitas Air per Kecamatan Pesisir Banten

In [ ]:
final_path_t1 = path_t1 if os.path.exists(path_t1) else os.path.join(COMPOSITE_DIR, f"Banten_{YEAR_T1}.tif")
final_path_t2 = path_t2 if os.path.exists(path_t2) else os.path.join(COMPOSITE_DIR, f"Banten_{YEAR_T2}.tif")

print(f"Menggunakan baseline composite: {final_path_t1}")
print(f"Menggunakan comparison composite: {final_path_t2}")

ds_t1 = load_composite_as_dataset(final_path_t1)
ds_t2 = load_composite_as_dataset(final_path_t2)

pixel_area_ha = compute_pixel_area_ha(ds_t2)

DISTRICT_CONTEXTS = {
    "pulomerak": {
        "context": "kawasan Pelabuhan Penyeberangan Merak yang sangat aktif serta berdekatan dengan zona industri galangan kapal dan PLTU Suralaya",
        "sources": ["aktivitas kapal feri", "limpasan industri pesisir", "buangan termal/sedimen PLTU"]
    },
    "ciwandan": {
        "context": "wilayah Pelabuhan Logistik Ciwandan dan pusat industri berat (pabrik baja, kimia, dan semen) Cilegon",
        "sources": ["buangan limbah industri berat", "bongkar muat kapal curah", "limpasan drainase industri"]
    },
    "citangkil": {
        "context": "zona industri kimia dan petrokimia yang terhubung langsung dengan garis pantai industri Cilegon",
        "sources": ["residu polutan kimiawi", "limpasan limbah cair industri", "aktivitas transportasi logistik laut"]
    },
    "grogol": {
        "context": "wilayah pesisir utara Cilegon yang padat aktivitas manufaktur logam dan logistik pelabuhan",
        "sources": ["limpasan sedimentasi", "buangan domestik perkotaan", "debu industri pesisir"]
    },
    "cibeber": {
        "context": "daerah aliran sungai urban Cilegon yang membawa sisa buangan domestik perkotaan ke pesisir",
        "sources": ["limbah domestik rumah tangga", "sampah plastik", "limpasan air hujan kota"]
    },
    "jombang": {
        "context": "aliran drainase pusat kota Cilegon dengan kepadatan penduduk tinggi",
        "sources": ["buangan detergen domestik", "sanitasi perkotaan", "limbah komersial mikro"]
    },
    "kasemen": {
        "context": "wilayah Pelabuhan Perikanan Karangantu dan area budidaya tambak pesisir Serang yang sangat luas",
        "sources": ["sisa pakan tambak udang/ikan", "limbah organik domestik pesisir", "aktivitas pasar ikan Karangantu"]
    },
    "anyar": {
        "context": "kawasan pariwisata pantai utama Banten dengan kepadatan hotel, resort, dan rekreasi pesisir",
        "sources": ["limbah domestik perhotelan", "aktivitas wisatawan", "sedimen dari sungai sekitar"]
    },
    "cinangka": {
        "context": "zona wisata pantai berpasir dengan aktivitas rekreasi laut dan perhotelan intensif",
        "sources": ["aktivitas wisata air", "limbah cair domestik", "limpasan pertanian dari hulu"]
    },
    "bojonegara": {
        "context": "pusat industri galangan kapal, manufaktur lepas pantai, dan dermaga logistik swasta (jetty)",
        "sources": ["tumpahan minyak ringan/oli kapal", "sedimentasi akibat reklamasi/pengerukan", "buangan industri galangan"]
    },
    "kramatwatu": {
        "context": "zona peralihan muara sungai dan industri berat Bojonegara",
        "sources": ["limpasan muara sungai", "sedimentasi lumpur", "buangan pelabuhan sekitar"]
    },
    "pontang": {
        "context": "wilayah muara Sungai Ciujung dengan area tambak tradisional yang sangat dominan",
        "sources": ["limpasan pertanian hulu", "sisa pupuk tambak", "sedimentasi lumpur Sungai Ciujung"]
    },
    "tirtayasa": {
        "context": "daerah muara Ciujung bagian hilir dan hutan mangrove tersisa",
        "sources": ["nutrien pertanian", "sedimen lumpur tebal", "limbah cair tambak"]
    },
    "tanara": {
        "context": "hilir Sungai Cidurian dengan limpasan pertanian intensif",
        "sources": ["pupuk urea/pestisida pertanian", "sedimentasi lumpur", "limbah rumah tangga pedesaan"]
    },
    "carita": {
        "context": "kawasan wisata pantai rekreasi dan cagar alam pesisir",
        "sources": ["limbah domestik pariwisata", "aktivitas perahu wisata", "limpasan sungai kecil"]
    },
    "labuan": {
        "context": "zona Pelabuhan Perikanan Labuan dan PLTU Banten 2 Labuan",
        "sources": ["aktivitas kapal nelayan dan bahan bakar solar", "limpasan pemukiman nelayan padat", "limbah air hangat PLTU"]
    },
    "panimbang": {
        "context": "wilayah pesisir Teluk Lada yang dikembangkan sebagai KEK Pariwisata Tanjung Lesung",
        "sources": ["pembangunan infrastruktur wisata", "sedimentasi lumpur Teluk Lada", "limpasan pertanian hulu"]
    },
    "sumur": {
        "context": "wilayah penyangga Taman Nasional Ujung Kulon yang menghadap ke Selat Sunda",
        "sources": ["suspensi pasir alami", "limpasan sungai liar hutan hujan", "aktivitas nelayan tradisional"]
    },
    "bayah": {
        "context": "wilayah pesisir Samudra Hindia dengan pelabuhan khusus semen (jetty) dan pertambangan batubara/pasir di hulu",
        "sources": ["sedimentasi debu tambang/semen", "erosi alami tebing pantai", "limpasan lumpur sungai"]
    },
    "wanasalam": {
        "context": "pusat Pelabuhan Perikanan Binuangeun dengan aktivitas nelayan lepas pantai",
        "sources": ["limbah organik Tempat Pelelangan Ikan (TPI)", "buangan bahan bakar solar kapal", "limpasan tambak udang sekitar"]
    },
    "cihara": {
        "context": "pesisir selatan terbuka dengan karakteristik gelombang besar Samudra Hindia",
        "sources": ["abrasi tebing alami", "sedimentasi sungai lokal", "turbulensi pasir akibat ombak"]
    },
    "panggarangan": {
        "context": "pesisir terbuka dengan aktivitas tambang batu bara tradisional/pasir di hulu",
        "sources": ["limpasan sedimen tambang rakyat", "abrasi pantai alami", "buangan domestik sungai"]
    }
}

def generate_explanation(kec_name, status, ndti, ndci, kab_kota, stats):
    name_lower = kec_name.lower()
    profile = DISTRICT_CONTEXTS.get(name_lower)
    if profile:
        context_text = f"Kecamatan {kec_name} merupakan {profile['context']}. "
        sources_text = f"Kondisi ini dipengaruhi oleh {', '.join(profile['sources'])}."
    else:
        context_text = f"Kecamatan {kec_name} terletak di wilayah pesisir {kab_kota}. "
        sources_text = "Kondisi ini dipengaruhi oleh aktivitas domestik dan limpasan permukaan sekitar perairan pesisir."

    industry_text = _build_industry_text(stats)

    if status == "TIDAK SEHAT":
        reason = f"Status Kualitas Air di {kec_name} diklasifikasikan sebagai **TIDAK SEHAT**. {context_text}"
        param_reasons = []
        if ndti > 0.05:
            param_reasons.append(f"tingkat kekeruhan air (NDTI: {ndti:.4f}) melebihi ambang batas aman 0.05 yang menandakan sedimentasi pantai yang tinggi")
        if ndci > 0.08:
            param_reasons.append(f"konsentrasi klorofil-a (NDCI: {ndci:.4f}) melampaui batas aman 0.08 yang mengindikasikan adanya blooming alga (eutrofikasi) akibat penumpukan zat hara/nutrien")
        if param_reasons:
            reason += "Hal ini terbukti secara ilmiah melalui analisis citra Sentinel-2 di mana " + " dan ".join(param_reasons) + ". "
        else:
            reason += "Hasil analisis menunjukkan akumulasi parameter fisik-kimiawi air (TSS/CDOM) melampaui baku mutu optimal. "
        reason += sources_text + industry_text
    elif status == "SEDANG":
        reason = f"Kualitas air pesisir di {kec_name} berada dalam kondisi **SEDANG**. {context_text}Meskipun parameter kekeruhan (NDTI: {ndti:.4f}) dan klorofil-a (NDCI: {ndci:.4f}) masih berada dalam tingkat toleransi wajar, tetap diperlukan pengawasan karena adanya kontribusi polusi dari {', '.join(profile['sources']) if profile else 'aktivitas antropogenik lokal'}.{industry_text}"
    else:
        reason = f"Kualitas air pesisir di {kec_name} diklasifikasikan sebagai **SEHAT** (Optimum). {context_text}Kondisi fisik perairan terpantau sangat bersih dengan kekeruhan rendah (NDTI: {ndti:.4f}) dan kadar klorofil-a (NDCI: {ndci:.4f}) yang seimbang, menunjukkan sirkulasi perairan yang baik serta minimnya dampak negatif dari {', '.join(profile['sources']) if profile else 'limbah domestik perkotaan'}.{industry_text}"
    return reason

results = []
for i, row in enumerate(coastal_gdf.itertuples()):
    name_3 = row.Kecamatan
    name_2 = row.Kabupaten_Kota
    geom = row.geometry
    
    print(f"[{i+1}/{len(coastal_gdf)}] Menganalisis Kecamatan {name_3} ({name_2})...")
    stats_t1 = analyze_kecamatan_water_quality(ds_t1, geom, pixel_area_ha)
    stats_t2 = analyze_kecamatan_water_quality(ds_t2, geom, pixel_area_ha)
    
    if stats_t2 is None:
        continue
    if stats_t1 is None:
        stats_t1 = {
            "water_area_ha": 0.0, "healthy_ha": 0.0, "moderate_ha": 0.0, "unhealthy_ha": 0.0,
            "pct_healthy": 0.0, "pct_moderate": 0.0, "pct_unhealthy": 0.0,
            "mean_ndti": 0.0, "mean_ndci": 0.0, "mean_tss": 0.0, "mean_cdom": 0.0, "status": "N/A"
        }
        
    diff_healthy = stats_t2["pct_healthy"] - stats_t1["pct_healthy"]
    diff_unhealthy = stats_t2["pct_unhealthy"] - stats_t1["pct_unhealthy"]
    
    if diff_healthy >= 5.0 and diff_unhealthy <= -5.0:
        tren = "MEMBAIK"
    elif diff_unhealthy >= 5.0:
        tren = "MEMBURUK"
    else:
        tren = "STABIL"
        
    # Hitung centroid dan jarak industri
    centroid = row.land_geom.centroid
    lat, lon = centroid.y, centroid.x
    stats_t2["centroid_latitude"] = round(lat, 6)
    stats_t2["centroid_longitude"] = round(lon, 6)
    nearest = find_nearest_industries(lat, lon, top_n=3)
    if nearest:
        stats_t2["industri_terdekat"] = nearest[0]["nama"]
        stats_t2["tipe_industri"] = nearest[0]["tipe"]
        stats_t2["jarak_industri_km"] = nearest[0]["jarak_km"]
        stats_t2["kategori_dampak_industri"] = get_dampak_category(nearest[0]["jarak_km"])
    if len(nearest) >= 2:
        stats_t2["industri_terdekat_2"] = nearest[1]["nama"]
        stats_t2["tipe_industri_2"] = nearest[1]["tipe"]
        stats_t2["jarak_industri_2_km"] = nearest[1]["jarak_km"]
    if len(nearest) >= 3:
        stats_t2["industri_terdekat_3"] = nearest[2]["nama"]
        stats_t2["tipe_industri_3"] = nearest[2]["tipe"]
        stats_t2["jarak_industri_3_km"] = nearest[2]["jarak_km"]

    explanation = generate_explanation(name_3, stats_t2["status"], stats_t2["mean_ndti"], stats_t2["mean_ndci"], name_2, stats_t2)
    results.append({
        "Kabupaten_Kota": name_2,
        "Kecamatan": name_3,
        "Luas_Air_2026_Ha": stats_t2["water_area_ha"],
        "Sehat_2026_Ha": stats_t2["healthy_ha"],
        "Sedang_2026_Ha": stats_t2["moderate_ha"],
        "TidakSehat_2026_Ha": stats_t2["unhealthy_ha"],
        "Pct_Sehat_2026": stats_t2["pct_healthy"],
        "Pct_Sedang_2026": stats_t2["pct_moderate"],
        "Pct_TidakSehat_2026": stats_t2["pct_unhealthy"],
        "Mean_NDTI_2026": stats_t2["mean_ndti"],
        "Mean_NDCI_2026": stats_t2["mean_ndci"],
        "Mean_TSS_2026": stats_t2["mean_tss"],
        "Mean_CDOM_2026": stats_t2["mean_cdom"],
        "Status_Kualitas_2026": stats_t2["status"],
        "centroid_latitude": stats_t2.get("centroid_latitude"),
        "centroid_longitude": stats_t2.get("centroid_longitude"),
        "industri_terdekat": stats_t2.get("industri_terdekat"),
        "tipe_industri": stats_t2.get("tipe_industri"),
        "jarak_industri_km": stats_t2.get("jarak_industri_km"),
        "kategori_dampak_industri": stats_t2.get("kategori_dampak_industri"),
        "industri_terdekat_2": stats_t2.get("industri_terdekat_2"),
        "tipe_industri_2": stats_t2.get("tipe_industri_2"),
        "jarak_industri_2_km": stats_t2.get("jarak_industri_2_km"),
        "industri_terdekat_3": stats_t2.get("industri_terdekat_3"),
        "tipe_industri_3": stats_t2.get("tipe_industri_3"),
        "jarak_industri_3_km": stats_t2.get("jarak_industri_3_km"),
        "Luas_Air_2017_Ha": stats_t1["water_area_ha"],
        "Sehat_2017_Ha": stats_t1["healthy_ha"],
        "Pct_Sehat_2017": stats_t1["pct_healthy"],
        "Mean_NDTI_2017": stats_t1["mean_ndti"],
        "Mean_NDCI_2017": stats_t1["mean_ndci"],
        "Status_Kualitas_2017": stats_t1["status"],
        "Delta_Pct_Sehat": round(diff_healthy, 1),
        "Tren_Kualitas": tren,
        "penjelasan_kualitas": explanation
    })

df_results = pd.DataFrame(results)
print(f"\nAnalisis selesai untuk {len(df_results)} kecamatan.")
df_results.head(10)

---
## 9. Pelatihan Model Machine Learning Global (Random Forest Kualitas Air Banten)

In [ ]:
def train_global_rf_model(ds, combined_geom):
    print("Melatih Model Random Forest Global untuk Kualitas Air Banten...")
    try:
        # Potong dengan batas perairan pesisir gabungan seluruh Banten
        clipped_ds = ds.rio.clip([combined_geom], crs="EPSG:4326", all_touched=True)
    except Exception as e:
        print(f"Error memotong composite global: {e}")
        return None, None
        
    clipped_ds = calculate_water_quality_indices(clipped_ds)
    threshold_classes = classify_water_pixels_threshold(clipped_ds)
    water_mask = threshold_classes > 0
    features_bands = ["B02", "B03", "B04", "B08", "B11"]
    if "B05" in clipped_ds:
        features_bands.append("B05")
    features_bands.extend(["NDTI", "NDCI", "TSS", "CDOM"])
    
    features_bands = [f for f in features_bands if f in clipped_ds]
    print(f"Fitur: {features_bands}")
    
    features_list = [clipped_ds[band].values[water_mask] for band in features_bands]
    X = np.stack(features_list, axis=1)
    y = threshold_classes[water_mask]
    
    valid = ~np.isnan(X).any(axis=1)
    X = X[valid]
    y = y[valid]
    
    print(f"Jumlah sampel piksel air global: {len(X)}")
    
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    
    rf = RandomForestClassifier(n_estimators=100, min_samples_leaf=5, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    
    y_pred = rf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    
    importances = rf.feature_importances_.tolist()
    feat_imp = {band: round(imp, 4) for band, imp in zip(features_bands, importances)}
    
    print(f"Akurasi Validasi Model: {acc:.2%}")
    print(f"Kappa Coefficient: {kappa:.4f}")
    print(f"Confusion Matrix:\n{cm}")
    print(f"Feature Importances:\n{feat_imp}")
    
    metrics = {
        "accuracy": round(float(acc), 4),
        "kappa": round(float(kappa), 4),
        "confusion_matrix": cm.tolist(),
        "feature_importances": feat_imp,
        "n_train_samples": int(len(X_train)),
        "n_test_samples": int(len(X_test)),
        "features_used": features_bands,
        "trained_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    return rf, metrics

rf_model, rf_metrics = train_global_rf_model(ds_t2, combined_geom)

if rf_model is not None:
    os.makedirs(os.path.join(OUTPUT_DIR, "model"), exist_ok=True)
    model_path = os.path.join(OUTPUT_DIR, "model", "water_quality_rf.joblib")
    joblib.dump(rf_model, model_path)
    print(f"Model Random Forest global disimpan ke: {model_path}")
    
    meta_path = os.path.join(OUTPUT_DIR, "model", "model_metadata.json")
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(rf_metrics, f, indent=2, ensure_ascii=False)
    print(f"Metadata model global disimpan ke: {meta_path}")

---
## 9b. Analisis Kualitas Air Pantai Pesisir Banten

In [ ]:
coastal_beaches_gdf = load_banten_coastal_beaches(land_geom_utm)

beach_results = []
for i, row in enumerate(coastal_beaches_gdf.itertuples()):
    beach_name = row.Pantai
    kec_name = row.Kecamatan
    kab_name = row.Kabupaten_Kota
    geom = row.geometry
    lat = row.latitude
    lon = row.longitude
    
    print(f"[{i+1}/{len(coastal_beaches_gdf)}] Menganalisis Pantai {beach_name} ({kec_name})...")
    stats_t1 = analyze_kecamatan_water_quality(ds_t1, geom, pixel_area_ha)
    stats_t2 = analyze_kecamatan_water_quality(ds_t2, geom, pixel_area_ha)
    
    if stats_t2 is None:
        continue
    if stats_t1 is None:
        stats_t1 = {
            "water_area_ha": 0.0, "healthy_ha": 0.0, "moderate_ha": 0.0, "unhealthy_ha": 0.0,
            "pct_healthy": 0.0, "pct_moderate": 0.0, "pct_unhealthy": 0.0,
            "mean_ndti": 0.0, "mean_ndci": 0.0, "mean_tss": 0.0, "mean_cdom": 0.0, "status": "N/A"
        }
        
    diff_healthy = stats_t2["pct_healthy"] - stats_t1["pct_healthy"]
    diff_unhealthy = stats_t2["pct_unhealthy"] - stats_t1["pct_unhealthy"]
    
    if diff_healthy >= 5.0 and diff_unhealthy <= -5.0:
        tren = "MEMBAIK"
    elif diff_unhealthy >= 5.0:
        tren = "MEMBURUK"
    else:
        tren = "STABIL"
        
    nearest = find_nearest_industries(lat, lon, top_n=3)
    stats_t2["latitude"] = lat
    stats_t2["longitude"] = lon
    if nearest:
        stats_t2["industri_terdekat"] = nearest[0]["nama"]
        stats_t2["tipe_industri"] = nearest[0]["tipe"]
        stats_t2["jarak_industri_km"] = nearest[0]["jarak_km"]
        stats_t2["kategori_dampak_industri"] = get_dampak_category(nearest[0]["jarak_km"])
    if len(nearest) >= 2:
        stats_t2["industri_terdekat_2"] = nearest[1]["nama"]
        stats_t2["tipe_industri_2"] = nearest[1]["tipe"]
        stats_t2["jarak_industri_2_km"] = nearest[1]["jarak_km"]
    if len(nearest) >= 3:
        stats_t2["industri_terdekat_3"] = nearest[2]["nama"]
        stats_t2["tipe_industri_3"] = nearest[2]["tipe"]
        stats_t2["jarak_industri_3_km"] = nearest[2]["jarak_km"]

    explanation = generate_beach_explanation(beach_name, kec_name, stats_t2["status"], stats_t2["mean_ndti"], stats_t2["mean_ndci"], stats_t2)
    beach_results.append({
        "Pantai": beach_name,
        "Kecamatan": kec_name,
        "Kabupaten_Kota": kab_name,
        "latitude": lat,
        "longitude": lon,
        "Luas_Air_2026_Ha": stats_t2["water_area_ha"],
        "Sehat_2026_Ha": stats_t2["healthy_ha"],
        "Sedang_2026_Ha": stats_t2["moderate_ha"],
        "TidakSehat_2026_Ha": stats_t2["unhealthy_ha"],
        "Pct_Sehat_2026": stats_t2["pct_healthy"],
        "Pct_Sedang_2026": stats_t2["pct_moderate"],
        "Pct_TidakSehat_2026": stats_t2["pct_unhealthy"],
        "Mean_NDTI_2026": stats_t2["mean_ndti"],
        "Mean_NDCI_2026": stats_t2["mean_ndci"],
        "Mean_TSS_2026": stats_t2["mean_tss"],
        "Mean_CDOM_2026": stats_t2["mean_cdom"],
        "Status_Kualitas_2026": stats_t2["status"],
        "industri_terdekat": stats_t2.get("industri_terdekat"),
        "tipe_industri": stats_t2.get("tipe_industri"),
        "jarak_industri_km": stats_t2.get("jarak_industri_km"),
        "kategori_dampak_industri": stats_t2.get("kategori_dampak_industri"),
        "industri_terdekat_2": stats_t2.get("industri_terdekat_2"),
        "tipe_industri_2": stats_t2.get("tipe_industri_2"),
        "jarak_industri_2_km": stats_t2.get("jarak_industri_2_km"),
        "industri_terdekat_3": stats_t2.get("industri_terdekat_3"),
        "tipe_industri_3": stats_t2.get("tipe_industri_3"),
        "jarak_industri_3_km": stats_t2.get("jarak_industri_3_km"),
        "Luas_Air_2017_Ha": stats_t1["water_area_ha"],
        "Sehat_2017_Ha": stats_t1["healthy_ha"],
        "Pct_Sehat_2017": stats_t1["pct_healthy"],
        "Mean_NDTI_2017": stats_t1["mean_ndti"],
        "Mean_NDCI_2017": stats_t1["mean_ndci"],
        "Status_Kualitas_2017": stats_t1["status"],
        "Delta_Pct_Sehat": round(diff_healthy, 1),
        "Tren_Kualitas": tren,
        "penjelasan_kualitas": explanation
    })

df_beach_results = pd.DataFrame(beach_results)
print(f"\nAnalisis selesai untuk {len(df_beach_results)} pantai.")
df_beach_results.head(10)

---
## 9c. Sistem Rekomendasi Pantai Tersehat (Multi-Criteria Weighted Health Score)

Pada bagian ini, kita menjalankan model sistem rekomendasi untuk pantai di pesisir Banten. Model ini mengalkulasi skor komposit **Health Score (0-100)** untuk setiap pantai berdasarkan parameter kualitas air dan faktor lingkungan dengan bobot multi-kriteria.

In [ ]:
# ==============================================================================
# SISTEM REKOMENDASI PANTAI TERSEHAT (Multi-Criteria Weighted Scoring)
# ==============================================================================

# Bobot parameter untuk skor komposit (total = 1.0)
WEIGHTS = {
    "pct_sehat": 0.30,       # Persentase area sehat
    "ndti_inv": 0.20,        # Kekeruhan (inverted)
    "ndci_inv": 0.10,        # Klorofil-a (inverted)
    "tss_inv": 0.10,         # TSS (inverted)
    "cdom_inv": 0.05,        # CDOM (inverted)
    "tren": 0.15,            # Tren kualitas historis
    "industri_inv": 0.10,    # Dampak industri terdekat (inverted)
}

TREN_SCORES = {"MEMBAIK": 1.0, "STABIL": 0.5, "MEMBURUK": 0.0}
INDUSTRI_SCORES = {"RENDAH": 1.0, "SEDANG": 0.5, "TINGGI": 0.0}

HEALTH_LABELS = [
    (80, "SANGAT DIREKOMENDASIKAN"),
    (60, "DIREKOMENDASIKAN"),
    (40, "CUKUP DIREKOMENDASIKAN"),
    (20, "KURANG DIREKOMENDASIKAN"),
    (0, "TIDAK DIREKOMENDASIKAN"),
]

def get_recommendation_label(score):
    for threshold, label in HEALTH_LABELS:
        if score >= threshold:
            return label
    return "TIDAK DIREKOMENDASIKAN"

# Kumpulkan nilai mentah untuk normalisasi
pct_vals = df_beach_results["Pct_Sehat_2026"].values
ndti_vals = df_beach_results["Mean_NDTI_2026"].abs().values
ndci_vals = df_beach_results["Mean_NDCI_2026"].abs().values
tss_vals = df_beach_results["Mean_TSS_2026"].values
cdom_vals = df_beach_results["Mean_CDOM_2026"].values

min_pct, max_pct = pct_vals.min(), pct_vals.max()
min_ndti, max_ndti = ndti_vals.min(), ndti_vals.max()
min_ndci, max_ndci = ndci_vals.min(), ndci_vals.max()
min_tss, max_tss = tss_vals.min(), tss_vals.max()
min_cdom, max_cdom = cdom_vals.min(), cdom_vals.max()

def normalize(val, min_v, max_v, invert=False):
    if max_v == min_v:
        return 50.0
    norm = (val - min_v) / (max_v - min_v)
    if invert:
        norm = 1.0 - norm
    return max(0.0, min(100.0, norm * 100.0))

health_scores = []
recommendation_labels = []

for idx, row in df_beach_results.iterrows():
    s_pct = normalize(row["Pct_Sehat_2026"], min_pct, max_pct)
    s_ndti = normalize(abs(row["Mean_NDTI_2026"]), min_ndti, max_ndti, invert=True)
    s_ndci = normalize(abs(row["Mean_NDCI_2026"]), min_ndci, max_ndci, invert=True)
    s_tss = normalize(row["Mean_TSS_2026"], min_tss, max_tss, invert=True)
    s_cdom = normalize(row["Mean_CDOM_2026"], min_cdom, max_cdom, invert=True)
    s_tren = TREN_SCORES.get(row["Tren_Kualitas"], 0.5) * 100.0
    s_industri = INDUSTRI_SCORES.get(row["kategori_dampak_industri"], 0.5) * 100.0
    
    score = (
        WEIGHTS["pct_sehat"] * s_pct +
        WEIGHTS["ndti_inv"] * s_ndti +
        WEIGHTS["ndci_inv"] * s_ndci +
        WEIGHTS["tss_inv"] * s_tss +
        WEIGHTS["cdom_inv"] * s_cdom +
        WEIGHTS["tren"] * s_tren +
        WEIGHTS["industri_inv"] * s_industri
    )
    score = round(score, 2)
    health_scores.append(score)
    recommendation_labels.append(get_recommendation_label(score))

df_beach_results["Health_Score"] = health_scores
df_beach_results["Label_Rekomendasi"] = recommendation_labels

# Urutkan berdasarkan Health Score tertinggi
df_beach_results = df_beach_results.sort_values(by="Health_Score", ascending=False).reset_index(drop=True)
df_beach_results["Ranking"] = df_beach_results.index + 1

print("=== TOP 5 PANTAI TERSEHAT (SISTEM REKOMENDASI) ===")
print(df_beach_results[["Ranking", "Pantai", "Kecamatan", "Health_Score", "Label_Rekomendasi"]].head(5).to_string(index=False))

---
## 10. Ekspor & Pemetaan Hasil Spasial

In [ ]:
# Simpan CSV hasil statistik kecamatan
os.makedirs(OUTPUT_DIR, exist_ok=True)
csv_path = os.path.join(OUTPUT_DIR, "banten_water_quality_kecamatan.csv")
df_results.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"Hasil statistik kecamatan disimpan ke CSV: {csv_path}")

# Simpan JSON per kecamatan
json_path = os.path.join(OUTPUT_DIR, "banten_water_quality_kecamatan.json")
json_dict = df_results.set_index("Kecamatan").to_dict(orient="index")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(json_dict, f, indent=2, ensure_ascii=False)
print(f"Hasil statistik kecamatan disimpan ke JSON: {json_path}")

# Simpan JSON Daftar Industri/Pabrik
industries_json_path = os.path.join(OUTPUT_DIR, "banten_industries.json")
with open(industries_json_path, "w", encoding="utf-8") as f:
    json.dump(INDUSTRIES, f, indent=2, ensure_ascii=False)
print(f"Daftar lokasi industri disimpan ke JSON: {industries_json_path}")

# Simpan CSV hasil statistik pantai
beach_csv_path = os.path.join(OUTPUT_DIR, "banten_water_quality_beach.csv")
df_beach_results.to_csv(beach_csv_path, index=False, encoding="utf-8-sig")
print(f"Hasil statistik pantai disimpan ke CSV: {beach_csv_path}")

# Simpan JSON per pantai
beach_json_path = os.path.join(OUTPUT_DIR, "banten_water_quality_beach.json")
beach_json_dict = df_beach_results.set_index("Pantai").to_dict(orient="index")
with open(beach_json_path, "w", encoding="utf-8") as f:
    json.dump(beach_json_dict, f, indent=2, ensure_ascii=False)
print(f"Hasil statistik pantai disimpan ke JSON: {beach_json_path}")

# Simpan Geometri Kecamatan & Pantai untuk Visualisasi Cepat
try:
    # Lakukan penyederhanaan geometri (simplify) untuk mengurangi ukuran file HTML dan mempercepat rendering peta
    coastal_water_gdf = coastal_gdf.set_geometry("geometry").copy()
    coastal_water_gdf["geometry"] = coastal_water_gdf["geometry"].simplify(0.0002, preserve_topology=True)
    coastal_water_gdf[["GID_3", "Kabupaten_Kota", "Kecamatan", "geometry"]].to_file(os.path.join(OUTPUT_DIR, "banten_coastal_kecamatan_water.geojson"), driver="GeoJSON")
    
    coastal_land_gdf = coastal_gdf.set_geometry("land_geom").copy()
    coastal_land_gdf["land_geom"] = coastal_land_gdf["land_geom"].simplify(0.0002, preserve_topology=True)
    coastal_land_gdf[["GID_3", "Kabupaten_Kota", "Kecamatan", "land_geom"]].rename_geometry("geometry").to_file(os.path.join(OUTPUT_DIR, "banten_coastal_kecamatan_land.geojson"), driver="GeoJSON")
    
    coastal_beaches_gdf_copy = coastal_beaches_gdf.copy()
    coastal_beaches_gdf_copy["geometry"] = coastal_beaches_gdf_copy["geometry"].simplify(0.0002, preserve_topology=True)
    coastal_beaches_gdf_copy[["Pantai", "Kecamatan", "Kabupaten_Kota", "latitude", "longitude", "geometry"]].to_file(os.path.join(OUTPUT_DIR, "banten_coastal_beaches.geojson"), driver="GeoJSON")
    print("Geometri pesisir & pantai berhasil disederhanakan dan disimpan ke GeoJSON!")
except Exception as e:
    print(f"Warning: Gagal menyimpan cache GeoJSON ({e})")

# Visualisasi Peta Interaktif Folium
def create_folium_visualization(coastal_gdf, results_df, output_path, beaches_gdf=None, beaches_results_df=None):
    merged_gdf = coastal_gdf.merge(results_df, on=["Kabupaten_Kota", "Kecamatan"], how="inner")
    centroid = merged_gdf.union_all().centroid
    map_center = [centroid.y, centroid.x]
    m = folium.Map(location=map_center, zoom_start=10, tiles="cartodbpositron")
    
    plugins.Fullscreen(position="topright", title="Fullscreen", title_cancel="Exit").add_to(m)
    color_map = {"SEHAT": "#2ecc71", "SEDANG": "#f1c40f", "TIDAK SEHAT": "#e74c3c"}
    
    # 1. Batas Darat Kecamatan
    land_layer = folium.FeatureGroup(name="Darat Kecamatan (Administratif)", show=True)
    for idx, row in merged_gdf.iterrows():
        folium.GeoJson(
            mapping(row["land_geom"]),
            style_function=lambda x: {
                "fillColor": "#bdc3c7", "color": "#7f8c8d", "weight": 1.0, "fillOpacity": 0.15
            },
            tooltip=f"{row['Kecamatan']}, {row['Kabupaten_Kota']}"
        ).add_to(land_layer)
    land_layer.add_to(m)
    
    # 2. Batas Air Pesisir (Warna Hijau/Kuning/Merah)
    water_layer = folium.FeatureGroup(name="Kualitas Air Pesisir (3 km)", show=True)
    for idx, row in merged_gdf.iterrows():
        status = row["Status_Kualitas_2026"]
        color = color_map.get(status, "#7f8c8d")
        tooltip_text = f"Kecamatan {row['Kecamatan']} ({status})"
        
        popup_html = f"""
        <div style='font-family: Arial, sans-serif; font-size: 13px; width: 300px; padding: 5px;'>
            <h4 style='margin: 0 0 5px 0; color: #2c3e50;'>Kecamatan {row['Kecamatan']}</h4>
            <span style='font-size: 11px; color: #7f8c8d;'>{row['Kabupaten_Kota']}</span>
            <hr style='margin: 8px 0;'/>
            <b>Luas Zona Air:</b> {row['Luas_Air_2026_Ha']:.1f} Ha<br/>
            <b>Health Score:</b> <span style='color: #2980b9; font-weight: bold;'>{row.get('Health_Score', 'N/A')}/100</span> ({row.get('Label_Rekomendasi', 'N/A')})<br/>
            <b>Status Kualitas (2026):</b> 
            <span style='color: {color}; font-weight: bold;'>{status}</span><br/>
            <b>Perubahan (2017 -> 2026):</b> 
            <span style='font-weight: bold;'>{row['Tren_Kualitas']}</span><br/>
            <hr style='margin: 8px 0;'/>
            <div style='font-size: 11px; line-height: 1.4; background-color: #f8f9fa; padding: 6px; border-left: 3px solid #3498db; margin: 5px 0;'>
                <b>Analisis Kondisi:</b><br/>{row.get('penjelasan_kualitas', 'Tidak ada data penjelasan.')}
            </div>
            <hr style='margin: 8px 0;'/>
            <table style='width: 100%; font-size: 12px;'>
                <tr style='background: #f8f9fa;'>
                    <td>🟢 <b>Sehat:</b></td>
                    <td style='text-align: right;'>{row['Pct_Sehat_2026']:.1f}% ({row['Sehat_2026_Ha']:.1f} Ha)</td>
                </tr>
                <tr>
                    <td>🟡 <b>Sedang:</b></td>
                    <td style='text-align: right;'>{row['Pct_Sedang_2026']:.1f}% ({row['Sedang_2026_Ha']:.1f} Ha)</td>
                </tr>
                <tr style='background: #f8f9fa;'>
                    <td>🔴 <b>Tidak Sehat:</b></td>
                    <td style='text-align: right;'>{row['Pct_TidakSehat_2026']:.1f}% ({row['TidakSehat_2026_Ha']:.1f} Ha)</td>
                </tr>
            </table>
        </div>
        """
        folium.GeoJson(
            mapping(row["geometry"]),
            style_function=lambda x, col=color: {
                "fillColor": col, "color": col, "weight": 1.5, "fillOpacity": 0.55
            },
            highlight_function=lambda x, col=color: {
                "fillOpacity": 0.85, "weight": 2.5
            },
            tooltip=tooltip_text
        ).add_child(folium.Popup(popup_html)).add_to(water_layer)
    water_layer.add_to(m)
    
    # 3. Layer Pantai Banten (Circle Markers & Buffer Polygons)
    if beaches_gdf is not None and beaches_results_df is not None:
        beach_layer = folium.FeatureGroup(name="Kualitas Air Pantai Banten (1 km)", show=True)
        merged_beaches = beaches_gdf.merge(beaches_results_df, on=["Pantai", "Kecamatan", "Kabupaten_Kota", "latitude", "longitude"], how="inner")
        
        for idx, row in merged_beaches.iterrows():
            geojson_beach_water = mapping(row["geometry"])
            status = row["Status_Kualitas_2026"]
            color = color_map.get(status, "#7f8c8d")
            tooltip_text = f"{row['Pantai']} ({status})"
            
            popup_html = f"""
            <div style='font-family: Arial, sans-serif; font-size: 13px; width: 300px; padding: 5px;'>
                <h4 style='margin: 0 0 5px 0; color: #2c3e50;'>{row['Pantai']}</h4>
                <span style='font-size: 11px; color: #7f8c8d;'>Kecamatan {row['Kecamatan']}, {row['Kabupaten_Kota']}</span>
                <hr style='margin: 8px 0;'/>
                <b>Health Score:</b> <span style='color: #2980b9; font-weight: bold;'>{row.get('Health_Score', 'N/A')}/100</span> ({row.get('Label_Rekomendasi', 'N/A')})<br/>
                <b>Status Kualitas (2026):</b> 
                <span style='color: {color}; font-weight: bold;'>{status}</span><br/>
                <b>Perubahan (2017 -> 2026):</b> 
                <span style='font-weight: bold;'>{row['Tren_Kualitas']}</span><br/>
                <hr style='margin: 8px 0;'/>
                <div style='font-size: 11px; line-height: 1.4; background-color: #f8f9fa; padding: 6px; border-left: 3px solid #3498db; margin: 5px 0;'>
                    <b>Analisis Kondisi:</b><br/>{row.get('penjelasan_kualitas', 'Tidak ada data penjelasan.')}
                </div>
                <hr style='margin: 8px 0;'/>
                <table style='width: 100%; font-size: 12px;'>
                    <tr style='background: #f8f9fa;'>
                        <td>🟢 <b>Sehat:</b></td>
                        <td style='text-align: right;'>{row['Pct_Sehat_2026']:.1f}%</td>
                    </tr>
                    <tr>
                        <td>🟡 <b>Sedang:</b></td>
                        <td style='text-align: right;'>{row['Pct_Sedang_2026']:.1f}%</td>
                    </tr>
                    <tr style='background: #f8f9fa;'>
                        <td>🔴 <b>Tidak Sehat:</b></td>
                        <td style='text-align: right;'>{row['Pct_TidakSehat_2026']:.1f}%</td>
                    </tr>
                </table>
                <hr style='margin: 8px 0;'/>
                <div style='font-size: 11px;'>
                    <b>Mean NDTI (Turbidity):</b> {row['Mean_NDTI_2026']:.4f}<br/>
                    <b>Mean NDCI (Chlorophyll):</b> {row['Mean_NDCI_2026']:.4f}<br/>
                    <b>Mean TSS Proxy:</b> {row['Mean_TSS_2026']:.4f}
                </div>
            </div>
            """
            
            marker_color = "green" if status == "SEHAT" else "orange" if status == "SEDANG" else "red"
            folium.Marker(
                location=[row["latitude"], row["longitude"]],
                icon=folium.Icon(color=marker_color, icon="info-sign"),
                tooltip=row["Pantai"],
                popup=folium.Popup(popup_html, max_width=320)
            ).add_to(beach_layer)
            
            folium.GeoJson(
                geojson_beach_water,
                style_function=lambda x, col=color: {
                    "fillColor": col, "color": col, "weight": 1.0, "fillOpacity": 0.35
                },
                highlight_function=lambda x, col=color: {
                    "fillOpacity": 0.65, "weight": 2.0
                },
                tooltip=tooltip_text
            ).add_child(folium.Popup(popup_html)).add_to(beach_layer)
            
        beach_layer.add_to(m)
    
    # Legenda
    legend_html = f"""
     <div style=\"position: fixed; 
                 bottom: 50px; left: 50px; width: 220px; height: 160px; 
                 border:2px solid grey; z-index:9999; font-size:12px;
                 background-color:white;
                 opacity: 0.9;
                 padding: 10px;
                 border-radius: 5px;
                 font-family: Arial, sans-serif;\">
     <h4 style=\"margin:0 0 8px 0; font-size:13px; color:#2c3e50;\">Kualitas Air Pesisir</h4>
     <div style=\"margin-bottom: 5px;\"><i style=\"background:#2ecc71; width:28px; height:12px; float:left; margin-right:8px; opacity:0.7; border-radius: 2px;\"></i>🟢 <b>SEHAT</b> (>50% Sehat)</div>
     <div style=\"margin-bottom: 5px;\"><i style=\"background:#f1c40f; width:28px; height:12px; float:left; margin-right:8px; opacity:0.7; border-radius: 2px;\"></i>🟡 <b>SEDANG</b> (Kondisi Sedang)</div>
     <div style=\"margin-bottom: 8px;\"><i style=\"background:#e74c3c; width:28px; height:12px; float:left; margin-right:8px; opacity:0.7; border-radius: 2px;\"></i>🔴 <b>TIDAK SEHAT</b> (>30% Unhealthy)</div>
     <hr style=\"margin: 5px 0;\"/>
     <span style=\"font-size:10px; color:#7f8c8d;\">Banten per Kecamatan & Pantai (Sentinel-2)</span>
     </div>
     """
    m.get_root().html.add_child(folium.Element(legend_html))
    folium.LayerControl().add_to(m)
    m.save(output_path)
    print(f"Peta interaktif disimpan ke: {output_path}")
    return m

html_path = os.path.join(OUTPUT_DIR, "banten_water_quality_map.html")
map_obj = create_folium_visualization(coastal_gdf, df_results, html_path, coastal_beaches_gdf, df_beach_results)
map_obj

---
## 11. Visualisasi Cepat Hasil Analisis (Tanpa Memproses Ulang Citra/openEO)

Jalankan cell di bawah ini untuk memuat data hasil analisis dari file JSON dan menampilkannya pada peta interaktif secara instan.

In [ ]:
# ==========================================
# CELL VISUALISASI CEPAT (OFFLINE)
# ==========================================

import os
import json
import pandas as pd
import geopandas as gpd
import folium
from folium import plugins
from shapely.geometry import mapping

OUTPUT_DIR = "output"

# Path file cache geometri GeoJSON
kecamatan_water_geojson = os.path.join(OUTPUT_DIR, "banten_coastal_kecamatan_water.geojson")
kecamatan_land_geojson = os.path.join(OUTPUT_DIR, "banten_coastal_kecamatan_land.geojson")
beaches_geojson = os.path.join(OUTPUT_DIR, "banten_coastal_beaches.geojson")

# Path file hasil analisis JSON
kecamatan_json_path = os.path.join(OUTPUT_DIR, "banten_water_quality_kecamatan.json")
beach_json_path = os.path.join(OUTPUT_DIR, "banten_water_quality_beach.json")

if not os.path.exists(kecamatan_json_path) or not os.path.exists(beach_json_path):
    print("WARNING: File JSON hasil analisis tidak ditemukan. Silakan jalankan analisis terlebih dahulu.")
else:
    # 1. Muat hasil analisis dari JSON
    with open(kecamatan_json_path, "r", encoding="utf-8") as f:
        kec_dict = json.load(f)
    df_results = pd.DataFrame.from_dict(kec_dict, orient="index").reset_index().rename(columns={"index": "Kecamatan"})
    
    with open(beach_json_path, "r", encoding="utf-8") as f:
        beach_dict = json.load(f)
    df_beach_results = pd.DataFrame.from_dict(beach_dict, orient="index").reset_index().rename(columns={"index": "Pantai"})
    
    # 2. Muat geometri dari GeoJSON cache
    if not (os.path.exists(kecamatan_water_geojson) and os.path.exists(kecamatan_land_geojson) and os.path.exists(beaches_geojson)):
        print("WARNING: File cache GeoJSON geometri tidak lengkap di folder output.")
        print("Menjalankan ekstraksi dari GADM (ini memerlukan file gadm41_IDN.gpkg)...")
        # Fallback ke ekstraksi jika cache belum dibuat
        GADM_PATH = "data/gadm41_IDN.gpkg"
        if not os.path.exists(GADM_PATH):
            raise FileNotFoundError(f"File GADM tidak ditemukan di {GADM_PATH}. Cache GeoJSON harus dibuat terlebih dahulu.")
        
        gdf = gpd.read_file(GADM_PATH, layer="ADM_ADM_3")
        banten = gdf[gdf["NAME_1"] == "Banten"].copy()
        neighbors = gdf[gdf["NAME_1"].isin(["Banten", "Jawa Barat", "Jakarta Raya", "Dki Jakarta"])].copy()
        
        banten_utm = banten.to_crs(epsg=32748)
        neighbors_utm = neighbors.to_crs(epsg=32748)
        land_geom_utm = neighbors_utm.union_all()
        
        coastal_rows = []
        for idx, row in banten_utm.iterrows():
            geom = row.geometry
            buffered = geom.buffer(3000)
            water_zone = buffered.difference(land_geom_utm)
            if water_zone.area / 10000.0 >= 10.0:
                coastal_rows.append({
                    "GID_3": row["GID_3"],
                    "Kabupaten_Kota": row["NAME_2"],
                    "Kecamatan": row["NAME_3"],
                    "land_geom": geom,
                    "geometry": water_zone
                })
        coastal_gdf = gpd.GeoDataFrame(coastal_rows, crs="EPSG:32748")
        coastal_gdf = coastal_gdf.to_crs(epsg=4326)
        coastal_gdf["land_geom"] = coastal_gdf["land_geom"].apply(
            lambda g: gpd.GeoSeries([g], crs="EPSG:32748").to_crs(epsg=4326).iloc[0]
        )
        
        # Ekstraksi Pantai
        beaches = [
            {"name": "Pantai Anyer", "Kecamatan": "Anyar", "Kabupaten_Kota": "Serang", "lat": -6.0465, "lon": 105.8850},
            {"name": "Pantai Carita", "Kecamatan": "Carita", "Kabupaten_Kota": "Pandeglang", "lat": -6.1305, "lon": 105.8427},
            {"name": "Pantai Tanjung Lesung", "Kecamatan": "Panimbang", "Kabupaten_Kota": "Pandeglang", "lat": -6.4785, "lon": 105.6565},
            {"name": "Pantai Sawarna", "Kecamatan": "Bayah", "Kabupaten_Kota": "Lebak", "lat": -6.9930, "lon": 106.3180},
            {"name": "Pantai Bagedur", "Kecamatan": "Malingping", "Kabupaten_Kota": "Lebak", "lat": -6.9038, "lon": 106.0125},
            {"name": "Pantai Karang Bolong", "Kecamatan": "Cinangka", "Kabupaten_Kota": "Serang", "lat": -6.1082, "lon": 105.8569},
            {"name": "Pantai Ciputih", "Kecamatan": "Sumur", "Kabupaten_Kota": "Pandeglang", "lat": -6.6575, "lon": 105.5180},
            {"name": "Pantai Pulau Umang", "Kecamatan": "Sumur", "Kabupaten_Kota": "Pandeglang", "lat": -6.64065, "lon": 105.58436},
            {"name": "Pantai Sambolo", "Kecamatan": "Anyar", "Kabupaten_Kota": "Serang", "lat": -6.0712, "lon": 105.8812},
            {"name": "Pantai Pasir Putih Sirih", "Kecamatan": "Anyar", "Kabupaten_Kota": "Serang", "lat": -6.0825, "lon": 105.8805},
            {"name": "Pantai Marbella", "Kecamatan": "Anyar", "Kabupaten_Kota": "Serang", "lat": -6.0620, "lon": 105.8825},
            {"name": "Pantai Florida Indah", "Kecamatan": "Cinangka", "Kabupaten_Kota": "Serang", "lat": -6.1345, "lon": 105.8670},
            {"name": "Pantai Jambu", "Kecamatan": "Cinangka", "Kabupaten_Kota": "Serang", "lat": -6.1158, "lon": 105.8640},
            {"name": "Pantai Lontar", "Kecamatan": "Tirtayasa", "Kabupaten_Kota": "Serang", "lat": -5.96884, "lon": 106.29646},
            {"name": "Pantai Tanjung Pasir", "Kecamatan": "Teluknaga", "Kabupaten_Kota": "Tangerang", "lat": -6.0150, "lon": 106.6850},
            {"name": "Pantai Tanjung Kait", "Kecamatan": "Mauk", "Kabupaten_Kota": "Tangerang", "lat": -6.0195, "lon": 106.4520},
            {"name": "Pantai Binuangeun", "Kecamatan": "Wanasalam", "Kabupaten_Kota": "Lebak", "lat": -6.8290, "lon": 105.9030},
            {"name": "Pantai Karang Taraje", "Kecamatan": "Bayah", "Kabupaten_Kota": "Lebak", "lat": -6.9912, "lon": 106.3312},
            {"name": "Pantai Sangiang", "Kecamatan": "Anyar", "Kabupaten_Kota": "Serang", "lat": -5.9535, "lon": 105.8565},
            {"name": "Pantai Pasir Putih Florida", "Kecamatan": "Cinangka", "Kabupaten_Kota": "Serang", "lat": -6.1265, "lon": 105.8645},
            {"name": "Pantai Karang Songsong", "Kecamatan": "Cihara", "Kabupaten_Kota": "Lebak", "lat": -6.88395, "lon": 106.11112},
            {"name": "Pantai Pulau Cangkir", "Kecamatan": "Kronjo", "Kabupaten_Kota": "Tangerang", "lat": -6.00889, "lon": 106.42000},
            {"name": "Pantai Pasir Putih Cihara", "Kecamatan": "Cihara", "Kabupaten_Kota": "Lebak", "lat": -6.84643, "lon": 106.06923},
            {"name": "Pantai Tanjung Layar", "Kecamatan": "Bayah", "Kabupaten_Kota": "Lebak", "lat": -6.99431, "lon": 106.30716},
            {"name": "Pantai Caringin", "Kecamatan": "Labuan", "Kabupaten_Kota": "Pandeglang", "lat": -6.35329, "lon": 105.82301}
        ]
        from shapely.geometry import Point
        df = pd.DataFrame(beaches)
        geometry = [Point(xy) for xy in zip(df['lon'], df['lat'])]
        beaches_gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")
        beaches_utm = beaches_gdf.to_crs(epsg=32748)
        
        coastal_beaches = []
        for idx, row in beaches_utm.iterrows():
            buffered = row.geometry.buffer(1000)
            water_zone = buffered.difference(land_geom_utm)
            if not water_zone.is_empty:
                coastal_beaches.append({
                    "Pantai": row["name"],
                    "Kecamatan": row["Kecamatan"],
                    "Kabupaten_Kota": row["Kabupaten_Kota"],
                    "latitude": row["lat"],
                    "longitude": row["lon"],
                    "geometry": water_zone
                })
        coastal_beaches_gdf = gpd.GeoDataFrame(coastal_beaches, crs="EPSG:32748")
        coastal_beaches_gdf = coastal_beaches_gdf.to_crs(epsg=4326)
        
        # Simpan ke GeoJSON cache dengan penyederhanaan geometry
        coastal_water_gdf = coastal_gdf.set_geometry("geometry").copy()
        coastal_water_gdf["geometry"] = coastal_water_gdf["geometry"].simplify(0.0002, preserve_topology=True)
        coastal_water_gdf[["GID_3", "Kabupaten_Kota", "Kecamatan", "geometry"]].to_file(kecamatan_water_geojson, driver="GeoJSON")
        
        coastal_land_gdf = coastal_gdf.set_geometry("land_geom").copy()
        coastal_land_gdf["land_geom"] = coastal_land_gdf["land_geom"].simplify(0.0002, preserve_topology=True)
        coastal_land_gdf[["GID_3", "Kabupaten_Kota", "Kecamatan", "land_geom"]].rename_geometry("geometry").to_file(kecamatan_land_geojson, driver="GeoJSON")
        
        coastal_beaches_gdf_copy = coastal_beaches_gdf.copy()
        coastal_beaches_gdf_copy["geometry"] = coastal_beaches_gdf_copy["geometry"].simplify(0.0002, preserve_topology=True)
        coastal_beaches_gdf_copy[["Pantai", "Kecamatan", "Kabupaten_Kota", "latitude", "longitude", "geometry"]].to_file(beaches_geojson, driver="GeoJSON")
    else:
        print("Memuat geometri dari cache GeoJSON (sangat cepat)... ")
        coastal_water_gdf = gpd.read_file(kecamatan_water_geojson)
        coastal_land_gdf = gpd.read_file(kecamatan_land_geojson)
        coastal_beaches_gdf = gpd.read_file(beaches_geojson)
        
        coastal_gdf = coastal_water_gdf.copy()
        coastal_gdf["land_geom"] = coastal_land_gdf["geometry"]
    
    # 3. Fungsi Visualisasi Peta (Direkonstruksi agar cell mandiri)
    def show_saved_map(coastal_gdf, results_df, output_path, beaches_gdf=None, beaches_results_df=None):
        merged_gdf = coastal_gdf.merge(results_df, on=["Kabupaten_Kota", "Kecamatan"], how="inner")
        centroid = merged_gdf.union_all().centroid
        map_center = [centroid.y, centroid.x]
        m = folium.Map(location=map_center, zoom_start=10, tiles="cartodbpositron")
        
        plugins.Fullscreen(position="topright", title="Fullscreen", title_cancel="Exit").add_to(m)
        color_map = {"SEHAT": "#2ecc71", "SEDANG": "#f1c40f", "TIDAK SEHAT": "#e74c3c"}
        
        # Darat
        land_layer = folium.FeatureGroup(name="Darat Kecamatan (Administratif)", show=True)
        for idx, row in merged_gdf.iterrows():
            folium.GeoJson(
                mapping(row["land_geom"]),
                style_function=lambda x: {
                    "fillColor": "#bdc3c7", "color": "#7f8c8d", "weight": 1.0, "fillOpacity": 0.15
                },
                tooltip=f"{row['Kecamatan']}, {row['Kabupaten_Kota']}"
            ).add_to(land_layer)
        land_layer.add_to(m)
        
        # Air
        water_layer = folium.FeatureGroup(name="Kualitas Air Pesisir (3 km)", show=True)
        for idx, row in merged_gdf.iterrows():
            status = row["Status_Kualitas_2026"]
            color = color_map.get(status, "#7f8c8d")
            tooltip_text = f"Kecamatan {row['Kecamatan']} ({status})"
            
            popup_html = f"""
            <div style='font-family: Arial, sans-serif; font-size: 13px; width: 300px; padding: 5px;'>
                <h4 style='margin: 0 0 5px 0; color: #2c3e50;'>Kecamatan {row['Kecamatan']}</h4>
                <span style='font-size: 11px; color: #7f8c8d;'>{row['Kabupaten_Kota']}</span>
                <hr style='margin: 8px 0;'/>
                <b>Luas Zona Air:</b> {row['Luas_Air_2026_Ha']:.1f} Ha<br/>
                <b>Health Score:</b> <span style='color: #2980b9; font-weight: bold;'>{row.get('Health_Score', 'N/A')}/100</span> ({row.get('Label_Rekomendasi', 'N/A')})<br/>
                <b>Status Kualitas (2026):</b> 
                <span style='color: {color}; font-weight: bold;'>{status}</span><br/>
                <b>Perubahan (2017 -> 2026):</b> 
                <span style='font-weight: bold;'>{row['Tren_Kualitas']}</span><br/>
                <hr style='margin: 8px 0;'/>
                <div style='font-size: 11px; line-height: 1.4; background-color: #f8f9fa; padding: 6px; border-left: 3px solid #3498db; margin: 5px 0;'>
                    <b>Analisis Kondisi:</b><br/>{row.get('penjelasan_kualitas', 'Tidak ada data penjelasan.')}
                </div>
                <hr style='margin: 8px 0;'/>
                <table style='width: 100%; font-size: 12px;'>
                    <tr style='background: #f8f9fa;'>
                        <td>🟢 <b>Sehat:</b></td>
                        <td style='text-align: right;'>{row['Pct_Sehat_2026']:.1f}% ({row['Sehat_2026_Ha']:.1f} Ha)</td>
                    </tr>
                    <tr>
                        <td>🟡 <b>Sedang:</b></td>
                        <td style='text-align: right;'>{row['Pct_Sedang_2026']:.1f}% ({row['Sedang_2026_Ha']:.1f} Ha)</td>
                    </tr>
                    <tr style='background: #f8f9fa;'>
                        <td>🔴 <b>Tidak Sehat:</b></td>
                        <td style='text-align: right;'>{row['Pct_TidakSehat_2026']:.1f}% ({row['TidakSehat_2026_Ha']:.1f} Ha)</td>
                    </tr>
                </table>
            </div>
            """
            folium.GeoJson(
                mapping(row["geometry"]),
                style_function=lambda x, col=color: {
                    "fillColor": col, "color": col, "weight": 1.5, "fillOpacity": 0.55
                },
                highlight_function=lambda x, col=color: {
                    "fillOpacity": 0.85, "weight": 2.5
                },
                tooltip=tooltip_text
            ).add_child(folium.Popup(popup_html)).add_to(water_layer)
        water_layer.add_to(m)
        
        # Pantai
        if beaches_gdf is not None and beaches_results_df is not None:
            beach_layer = folium.FeatureGroup(name="Kualitas Air Pantai Banten (1 km)", show=True)
            merged_beaches = beaches_gdf.merge(beaches_results_df, on=["Pantai", "Kecamatan", "Kabupaten_Kota", "latitude", "longitude"], how="inner")
            
            for idx, row in merged_beaches.iterrows():
                geojson_beach_water = mapping(row["geometry"])
                status = row["Status_Kualitas_2026"]
                color = color_map.get(status, "#7f8c8d")
                tooltip_text = f"{row['Pantai']} ({status})"
                
                popup_html = f"""
                <div style='font-family: Arial, sans-serif; font-size: 13px; width: 300px; padding: 5px;'>
                    <h4 style='margin: 0 0 5px 0; color: #2c3e50;'>{row['Pantai']}</h4>
                    <span style='font-size: 11px; color: #7f8c8d;'>Kecamatan {row['Kecamatan']}, {row['Kabupaten_Kota']}</span>
                    <hr style='margin: 8px 0;'/>
                    <b>Health Score:</b> <span style='color: #2980b9; font-weight: bold;'>{row.get('Health_Score', 'N/A')}/100</span> ({row.get('Label_Rekomendasi', 'N/A')})<br/>
                    <b>Status Kualitas (2026):</b> 
                    <span style='color: {color}; font-weight: bold;'>{status}</span><br/>
                    <b>Perubahan (2017 -> 2026):</b> 
                    <span style='font-weight: bold;'>{row['Tren_Kualitas']}</span><br/>
                    <hr style='margin: 8px 0;'/>
                    <div style='font-size: 11px; line-height: 1.4; background-color: #f8f9fa; padding: 6px; border-left: 3px solid #3498db; margin: 5px 0;'>
                        <b>Analisis Kondisi:</b><br/>{row.get('penjelasan_kualitas', 'Tidak ada data penjelasan.')}
                    </div>
                    <hr style='margin: 8px 0;'/>
                    <table style='width: 100%; font-size: 12px;'>
                        <tr style='background: #f8f9fa;'>
                            <td>🟢 <b>Sehat:</b></td>
                            <td style='text-align: right;'>{row['Pct_Sehat_2026']:.1f}%</td>
                        </tr>
                        <tr>
                            <td>🟡 <b>Sedang:</b></td>
                            <td style='text-align: right;'>{row['Pct_Sedang_2026']:.1f}%</td>
                        </tr>
                        <tr style='background: #f8f9fa;'>
                            <td>🔴 <b>Tidak Sehat:</b></td>
                            <td style='text-align: right;'>{row['Pct_TidakSehat_2026']:.1f}%</td>
                        </tr>
                    </table>
                    <hr style='margin: 8px 0;'/>
                    <div style='font-size: 11px;'>
                        <b>Mean NDTI:</b> {row['Mean_NDTI_2026']:.4f}<br/>
                        <b>Mean NDCI:</b> {row['Mean_NDCI_2026']:.4f}<br/>
                        <b>Mean TSS Proxy:</b> {row['Mean_TSS_2026']:.4f}
                    </div>
                </div>
                """
                
                marker_color = "green" if status == "SEHAT" else "orange" if status == "SEDANG" else "red"
                folium.Marker(
                    location=[row["latitude"], row["longitude"]],
                    icon=folium.Icon(color=marker_color, icon="info-sign"),
                    tooltip=row["Pantai"],
                    popup=folium.Popup(popup_html, max_width=320)
                ).add_to(beach_layer)
                
                folium.GeoJson(
                    geojson_beach_water,
                    style_function=lambda x, col=color: {
                        "fillColor": col, "color": col, "weight": 1.0, "fillOpacity": 0.35
                    },
                    highlight_function=lambda x, col=color: {
                        "fillOpacity": 0.65, "weight": 2.0
                    },
                    tooltip=tooltip_text
                ).add_child(folium.Popup(popup_html)).add_to(beach_layer)
                
            beach_layer.add_to(m)
        
        # Legenda
        legend_html = f"""
         <div style=\"position: fixed; 
                     bottom: 50px; left: 50px; width: 220px; height: 160px; 
                     border:2px solid grey; z-index:9999; font-size:12px;
                     background-color:white;
                     opacity: 0.9;
                     padding: 10px;
                     border-radius: 5px;
                     font-family: Arial, sans-serif;\">
         <h4 style=\"margin:0 0 8px 0; font-size:13px; color:#2c3e50;\">Kualitas Air Pesisir</h4>
         <div style=\"margin-bottom: 5px;\"><i style=\"background:#2ecc71; width:28px; height:12px; float:left; margin-right:8px; opacity:0.7; border-radius: 2px;\"></i>🟢 <b>SEHAT</b> (>50% Sehat)</div>
         <div style=\"margin-bottom: 5px;\"><i style=\"background:#f1c40f; width:28px; height:12px; float:left; margin-right:8px; opacity:0.7; border-radius: 2px;\"></i>🟡 <b>SEDANG</b> (Kondisi Sedang)</div>
         <div style=\"margin-bottom: 8px;\"><i style=\"background:#e74c3c; width:28px; height:12px; float:left; margin-right:8px; opacity:0.7; border-radius: 2px;\"></i>🔴 <b>TIDAK SEHAT</b> (>30% Unhealthy)</div>
         <hr style=\"margin: 5px 0;\"/>
         <span style=\"font-size:10px; color:#7f8c8d;\">Banten per Kecamatan & Pantai (Sentinel-2)</span>
         </div>
         """
        m.get_root().html.add_child(folium.Element(legend_html))
        folium.LayerControl().add_to(m)
        m.save(output_path)
        print(f"Peta interaktif disimpan ke: {output_path}")
        return m
    
    html_path = os.path.join(OUTPUT_DIR, "banten_water_quality_map.html")
    map_obj = show_saved_map(coastal_gdf, df_results, html_path, coastal_beaches_gdf, df_beach_results)

if 'map_obj' in locals():
    from IPython.display import display
    display(map_obj)
